<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>

# Explore ASAP CRN Data with R

Welcome to the **ASAP-CRN Learning Lab R Sample Notebook**.

This notebook helps you take your first steps into the ASAP CRN Cloud research environment using **R**. You will learn how to orient yourself in **Verily Workbench**, locate mounted CRN data resources, inspect harmonized metadata, and preview curated workflow outputs.

The goal is to help you move from workspace setup to research exploration with confidence. By the end, you will understand how CRN resources are organized and how to begin asking dataset-level questions using metadata, quality-control files, and single-cell data objects.

> **Tip:** Run each cell in order for the smoothest setup experience.  
> After completing the walkthrough, revisit the notebook to adapt it for your own research questions.


In [ ]:
# Required packages for this notebook
cran_packages <- c(
  "dplyr",
  "ggplot2",
  "readr",
  "fs",
  "Matrix",
  "IRdisplay"
)

bioc_packages <- c(
  "zellkonverter",
  "SingleCellExperiment"
)

# Install missing CRAN packages
missing_cran <- cran_packages[!sapply(cran_packages, requireNamespace, quietly = TRUE)]

if (length(missing_cran) > 0) {
  install.packages(missing_cran)
}

# Install BiocManager if needed
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

# Install missing Bioconductor packages
missing_bioc <- bioc_packages[!sapply(bioc_packages, requireNamespace, quietly = TRUE)]

if (length(missing_bioc) > 0) {
  BiocManager::install(missing_bioc, ask = FALSE, update = FALSE)
}

# Load required packages quietly
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(readr)
  library(fs)
  library(Matrix)
  library(IRdisplay)
  library(zellkonverter)
  library(SingleCellExperiment)
})

options(
  warn = -1,
  dplyr.width = Inf,
  repr.matrix.max.rows = 20,
  repr.matrix.max.cols = 20
)

message("R environment ready.")

## Table of Contents

1. Workspace Orientation
2. Select a Dataset Resource
   - 2.1 Set Resource Paths
3. Preview Harmonized Metadata
4. Inspect Curated Files
5. Explore the Dataset
   - 5.1 Inspect QC Plots
   - 5.2 Copy Data into Workspace Scratch Space
   - 5.3 Explore Cell-level Metadata
   - 5.4 Explore the AnnData Object in R
6. Reproducibility Notes
7. Provenance
8. Next Steps

## 1. Workspace Orientation

In the **ASAP-CRN Learning Lab** workspace, data and resources are mounted under your home directory, typically:

- `~/workspace/` – workspace mount for data and outputs  
- `~/workspace/*/asap-curated-team/` – team-specific curated and derived datasets
- `~/workspace/*/asap-curated-cohort/` – multi-team curated and derived datasets  
- `~/workspace/ws_files/` – your personal scratch space for files and results  

General subfolders: 
- `~/cohort_analysis` - Processed cohort-level outputs
- `~/preprocess` - Intermediate files from data curation outputs
In this section, we’ll confirm these paths and see what’s available.

In [ ]:
HOME_PATH <-  fs::path_home()
WS_ROOT <-  fs::path(HOME_PATH, "workspace")
WS_FILES <-  fs::path(WS_ROOT, "ws_files")

cat("Home directory:   ", HOME_PATH, "\n")
cat("Workspace root:   ", WS_ROOT, "\n")
cat("Workspace files:  ", WS_FILES, "\n")

if (!dir_exists(WS_ROOT)) {
  cat("Workspace mount not found. Attempting to mount workspace resources...\n")
  system("wb resource mount")
}

if (!dir_exists(WS_FILES)) {
  dir_create(WS_FILES, recurse = TRUE)
}

cat("\nWorkspace contents:\n")
dir_ls(WS_ROOT, type = "directory", recurse = FALSE) |>
  path_file() |>
  sort() |>
  paste0("/") |>
  cat(sep = "\n")

## 2. Select a Dataset Resource

For the next steps, we will work with datasets processed using the **PMDBS scRNAseq** workflow. Specifically, we will focus on the **cohort-level dataset**: `asap-cohort-pmdbs-sc-rnaseq`.  

This dataset represents a multi-dataset integration: samples from **multiple contributing datasets**, processed, curated, and harmonized into a single cohort resource.

### 2.1 Setting Resource Paths
The dataset paths follow a structured hierarchy. Each component has a specific meaning:

- **`workflow`** — identifies the workflow used for aggregation and integration.  
  Here we use the **[PMDBS scRNAseq workflow](https://github.com/ASAP-CRN/pmdbs-sc-rnaseq-wf)**.

- **`dataset_team`** — identifies the contributing team or grouping of datasets. For cohort-level analyses, this value is **`cohort`**, indicating multiple datasets combined.

- **`source`** — describes the biological source of the samples.  
  In this case, **`pmdbs`** refers to *post-mortem–derived brain samples*.

- **`dataset_type`** — describes the type of data generated.  
  Here it is **`sc-rnaseq`**, indicating single-cell RNA sequencing.

- **`bucket_name`** — the Google Cloud Storage bucket containing the curated dataset.

- **`dataset_name`** — a unique identifier for each curated dataset or collection.

In [ ]:
DATASETS_PATH <-  fs::path(WS_ROOT, "01_PMDBS", "pmdbs-sc-rnaseq-v3")

workflow <- "pmdbs_sc_rnaseq"
dataset_team <- "cohort"
dataset_source <- "pmdbs"
dataset_type <- "sc-rnaseq"

bucket_name <- paste(dataset_team, dataset_source, dataset_type, sep = "-")
dataset_name <- paste(dataset_team, dataset_source, dataset_type, sep = "-")
dataset_path <-  fs::path(DATASETS_PATH, bucket_name, workflow)

cat("Dataset name:", dataset_name, "\n")
cat("Dataset path:", dataset_path, "\n")

if (!dir_exists(dataset_path)) {
  stop(
    "Dataset path not found: ", dataset_path, "\n",
    "Confirm that the expected data collection is mounted in this workspace."
  )
}

## 3. Preview Harmonized Metadata

CRN metadata tables for each dataset are stored within the `release_resources` directory. 
These tables describe studies, protocols, subjects, samples, assays, conditions, and data files using harmonized CRN Common Data Elements. See the [Data Dictionary](https://storage.googleapis.com/asap-public-assets/wayfinding/ASAP-CRN-Cloud-Data-Dictionary.pdf) for an overview of the metadata tables. 

In this section, we preview one metadata table to confirm that the resource is available and readable.

> **Note:**  
> Metadata files are organized using the **short `dataset_name`**, not the `bucket_name`.  
> You can always use the File Browser tab of the side panel to explore directories and right-click any folder or file to copy its full path.


In [ ]:
ds_metadata_path <-  fs::path(WS_ROOT, "release_resources", dataset_name, "metadata")

if (!dir_exists(ds_metadata_path)) {
  stop(
    "Metadata path not found: ", ds_metadata_path, "\n",
    "Use the JupyterLab file browser to confirm the mounted release_resources structure."
  )
}

cat("Available metadata tables:\n")
dir_ls(ds_metadata_path, glob = "*.csv") |>
  path_file() |>
  sort() |>
  cat(sep = "\n")

In [ ]:
condition_df <- read_csv(
   fs::path(ds_metadata_path, "CONDITION.csv"),
  show_col_types = FALSE
)

head(condition_df, 10)

## 4. Inspect Curated Files

Now that the resource paths are defined, we can inspect the curated files available in the `cohort_analysis` directory.

In [ ]:
cohort_analysis_path <-  fs::path(dataset_path, "cohort_analysis")

if (!dir_exists(cohort_analysis_path)) {
  stop("cohort_analysis directory not found: ", cohort_analysis_path)
}

cat("Contents of cohort_analysis:\n")
dir_ls(cohort_analysis_path, recurse = FALSE) |>
  path_file() |>
  sort() |>
  cat(sep = "\n")

## 5. Explore the Dataset

With the directory structure in place, we can begin exploring the processed outputs produced by the PMDBS scRNA-seq workflow.

### 5.1 Inspect QC Plots

The curated dataset includes several **QC violin plots** summarizing key metrics (e.g., doublet score, gene counts, mitochondrial content). Let’s load and display all violin plot images found in the folder.

In [ ]:
# Inspect QC violin plots
qc_plot_paths <- dir_ls(
  cohort_analysis_path,
  glob = "*.violin.png",
  recurse = FALSE
)

if (length(qc_plot_paths) == 0) {
  cat("No violin plots found in cohort_analysis.\n")
} else {
  for (plot_path in qc_plot_paths) {
    cat("\n", path_file(plot_path), "\n")
    IRdisplay::display_png(file = plot_path)
  }
}

### 5.2 Copy Data into Workspace Scratch Space

In this step, we’ll set up a local directory inside our JupyterLab environment to store data files.  The `ws_files` area is a scratch space tied to your workspace — anything saved here can be accessed later in the notebook, processed with Python, or uploaded back to a workspace bucket for sharing.

We’ll create a folder called `workshop_files` under our workspace path (`WS_PATH`).  This ensures that all downloaded datasets are organized in one place.

In [ ]:
local_data_path <-  fs::path(WS_FILES, "pilot_workshop_files")

if (!dir_exists(local_data_path)) {
  dir_create(local_data_path, recurse = TRUE)
}

cat("Local data directory ready at:", local_data_path, "\n")

In [ ]:
copy_if_missing <- function(source_path, destination_path) {
  if (!file_exists(source_path)) {
    stop("Source file not found: ", source_path)
  }
  
  if (file_exists(destination_path)) {
    cat("Already exists:", path_file(destination_path), "\n")
  } else {
    file_copy(source_path, destination_path)
    cat("Copied:", path_file(destination_path), "\n")
  }
}

In [ ]:
cell_metadata_source <-  fs::path(
  cohort_analysis_path,
  paste0("asap-", dataset_team, ".final_metadata.csv")
)

cell_metadata_local_path <-  fs::path(
  local_data_path,
  path_file(cell_metadata_source)
)

adata_source <-  fs::path(
  cohort_analysis_path,
  paste0("asap-", dataset_team, ".final.h5ad")
)

adata_local_path <-  fs::path(
  local_data_path,
  path_file(adata_source)
)

copy_if_missing(cell_metadata_source, cell_metadata_local_path)
copy_if_missing(adata_source, adata_local_path)

### 5.3 Explore Cell-level Metadata

Once the data is available in our local `workshop_files` directory, we can begin exploring its metadata.

This field provides a compact entry point into the dataset’s metadata, making it easier to explore without handling the full expression matrix.

Each row in `obs` corresponds to a single cell (identified by a unique *barcode*) and contains:

- **Quality control metrics**: e.g. CellBender `cell_probability`, `n_genes_by_counts`, `total_counts`.  
- **Dataset references**: e.g. `sample`, `batch`, `team`, `dataset`.  
- **Downstream analysis results**: e.g. `UMAP_1`, `UMAP_2`, CellAssign `cell_type`, and Leiden cluster assignments.  

Together, these annotations summarize the key observations for each cell and provide a rich foundation for both quality assessment and downstream biological interpretation..


In [ ]:
cell_metadata_df <- read_csv(
  cell_metadata_local_path,
  show_col_types = FALSE
)

cat(
  "Loaded cell metadata for",
  format(nrow(cell_metadata_df), big.mark = ","),
  "cells\n"
)

cat(
  "Number of columns:",
  format(ncol(cell_metadata_df), big.mark = ","),
  "\n"
)

colnames(cell_metadata_df)

In [ ]:
cell_metadata_df |>
  dplyr::count(cell_type, sort = TRUE)

In [ ]:
set.seed(42)

plot_df <- cell_metadata_df |>
  sample_n(size = min(100000, nrow(cell_metadata_df))) |>
  mutate(
    n_genes_by_counts = as.numeric(n_genes_by_counts),
    total_counts = as.numeric(total_counts)
  )

ggplot(
  plot_df,
  aes(
    x = n_genes_by_counts,
    y = total_counts,
    color = cell_type
  )
) +
  geom_point(size = 0.3, alpha = 0.3) +
  facet_wrap(~ cell_type, scales = "free") +
  labs(
    x = "Number of genes detected",
    y = "Total counts",
    title = "QC metric relationship across sampled cells"
  ) +
  theme_minimal() +
  theme(legend.position = "none")

### 5.4 Explore the AnnData Object

In R, AnnData `.h5ad` files can be read using Bioconductor-supported tools. This notebook uses `zellkonverter` to load the `.h5ad` file and represent it as a `SingleCellExperiment` object, a common structure for single-cell analysis in R.  


In [ ]:
# Required package for native R AnnData / H5AD support
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

if (!requireNamespace("anndataR", quietly = TRUE)) {
  BiocManager::install("anndataR", ask = FALSE, update = FALSE)
}

if (!requireNamespace("rhdf5", quietly = TRUE)) {
  BiocManager::install("rhdf5", ask = FALSE, update = FALSE)
}


suppressPackageStartupMessages({
  library(anndataR)
})

In [ ]:
# Load the curated AnnData object into R
adata <- anndataR::read_h5ad(as.character(adata_local_path))

adata

In [ ]:
adata$shape
adata$obs_keys()
adata$var_keys()
adata$obsm_keys()

In [ ]:
# Pull UMAP coordinates
umap <- as.data.frame(adata$obsm[["X_umap"]])
colnames(umap)[1:2] <- c("UMAP_1", "UMAP_2")

# Pull cell metadata
obs <- as.data.frame(adata$obs)

# Combine coordinates and metadata
plot_df <- dplyr::bind_cols(umap, obs)

# Sample for faster plotting
set.seed(42)
plot_df_sample <- plot_df |>
  dplyr::sample_n(size = min(100000, nrow(plot_df)))

ggplot(plot_df_sample, aes(x = UMAP_1, y = UMAP_2, color = cell_type)) +
  geom_point(size = 0.2, alpha = 0.4) +
  labs(
    title = "UMAP colored by cell type",
    x = "UMAP 1",
    y = "UMAP 2",
    color = "Cell type"
  ) +
  theme_minimal()

In [ ]:
ggplot(plot_df_sample, aes(x = UMAP_1, y = UMAP_2, color = phase)) +
  geom_point(size = 0.2, alpha = 0.4) +
  labs(
    title = "UMAP colored by cell-cycle phase",
    x = "UMAP 1",
    y = "UMAP 2",
    color = "Phase"
  ) +
  theme_minimal()

In [ ]:
# System information
system("date")
cat("R version:", R.version.string, "\n")
cat("Number of CPU cores:", parallel::detectCores(), "\n")
system("grep '^MemTotal:' /proc/meminfo")

## 7. Provenance

This notebook was generated as part of the ASAP-CRN Verily Workbench Learning Lab to demonstrate workspace orientation using harmonized single-nucleus RNA-seq data from the ASAP Collaborative Research Network.

### Software & Environment

- Platform: Verily Workbench
- Runtime: Cloud app environment - JupyterLab with R
- Key libraries: dplyr, ggplot2, readr, fs, Matrix, zellkonverter, SingleCellExperiment
- Data format: CSV metadata tables and AnnData `.h5ad`

### Source Data

The dataset explored here originates from the integrated ASAP CRN post-mortem brain cohort:

- Curation workflow: PMDBS scRNA-seq pipeline
- Data type: cohort-level harmonized single-nucleus RNA-seq
- Input samples: multiple contributing datasets aggregated into a shared cohort resource
- Data storage: `workspace/01_PMDBS/pmdbs-sc-rnaseq-v3/cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq/cohort_analysis`

### Notebook-Generated Outputs

This notebook creates local working copies under:

`ws_files/pilot_workshop_files/`

## 8. Next Steps

In this notebook, you:

- Set up your workspace environment
- Located and previewed curated ASAP CRN data resources
- Reviewed harmonized metadata and cohort-level analysis outputs
- Loaded example files for downstream exploration in R

You are now ready to move from orientation to exploration.

Explore the [ASAP-CRN Learning Lab GitHub repository](https://github.com/ASAP-CRN/asap-crn-learning-lab.git) for more tutorials and example workflows.